# GPT-SoVITS Improve — Kaggle Grokking 연구

기존 Fast Minimal 설치 노트북의 연구 브랜치 버전입니다. Dataset `pymood1/w-voice-file`, GPU T4 ×2, Internet ON 설정 후 Run All. 전처리와 학습 설정은 Git의 `tools/kaggle_grokking_runner.py`에서 관리합니다. 연구 브랜치의 최신 커밋을 불러오므로 기존 노트북에서 이후 학습 코드 수정은 필요하지 않습니다.


## 0. Dataset 연결만 하면 됩니다

현재 기본 우선순위는 `pymood1/w-voice-file`입니다. 저장소가 다음 경로들을 자동 탐색합니다.

```text
/kaggle/input/datasets/pymood1/w-voice-file/datas/
/kaggle/input/w-voice-file/datas/
```

둘 다 없으면 `/kaggle/input` 아래에서 음성이 들어 있는 `datas/`를 자동 탐색합니다.
`training.list`는 선택 사항입니다. **이 노트북에서 경로 코드를 수정할 필요가 없습니다.**


### 데이터 준비는 Git 코드가 담당

이전 버전의 긴 Dataset 탐색/압축해제/`training.list` 변환 셀은 제거했습니다.
`tools/kaggle_dataset.py`가 Dataset을 준비하고, `tools/kaggle_launch_webui.py`가 Preflight와 WebUI 실행까지 연결합니다.


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import time


REPO_URL = "https://github.com/pymood1/GPT-SoVITS-improve.git"
REPO_DIR = Path("/kaggle/tmp/GPT-SoVITS")
RESEARCH_BRANCH = "research/grokking-gpt-sovits-20261121"


print("[1/2] GPT-SoVITS-improve 최신 연구 브랜치 준비", flush=True)

start = time.perf_counter()

if (REPO_DIR / ".git").exists():
    print("[git] 기존 repo 유지 + 최신 연구 브랜치 동기화", flush=True)

    subprocess.run(
        ["git", "fetch", "--depth", "1", "origin", RESEARCH_BRANCH],
        cwd=REPO_DIR,
        check=True,
    )

    subprocess.run(
        ["git", "reset", "--hard", "FETCH_HEAD"],
        cwd=REPO_DIR,
        check=True,
    )

else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)

    print("[git] 새 repo shallow clone", flush=True)

    clone_cmd = [
        "git",
        "clone",
        "--depth",
        "1",
        "--single-branch",
        "--branch",
        RESEARCH_BRANCH,
        REPO_URL,
        str(REPO_DIR),
    ]

    subprocess.run(clone_cmd, check=True)


elapsed = time.perf_counter() - start
print(f"[1/2] repo 준비 완료: {elapsed:.1f}s", flush=True)


print("[2/2] 전체 의존성 + pretrained 설치 시작", flush=True)

setup_script = REPO_DIR / "tools" / "colab_fast_setup.py"

# upstream Colab 설치 로직은 그대로 사용하고,
# Colab 전용 작업 경로만 Kaggle working 경로로 최소 치환합니다.
setup_text = setup_script.read_text(encoding="utf-8")
setup_text = setup_text.replace("/content/", "/kaggle/tmp/")
setup_script.write_text(setup_text, encoding="utf-8")


print("[check] setup script 문법 검사", flush=True)

subprocess.run(
    [sys.executable, "-m", "py_compile", str(setup_script)],
    check=True,
)


print("[run] setup script 실행 — 최근 로그만 표시", flush=True)

from collections import deque
from IPython.display import clear_output

setup_log = Path("/kaggle/tmp/gpt_sovits_setup.log")
tail = deque(maxlen=24)
pending = ""
last_render = 0.0


def render_tail(final=False):
    global last_render
    now = time.monotonic()
    if not final and now - last_render < 0.8:
        return
    last_render = now
    clear_output(wait=True)
    print("[1/2] repo 준비 완료")
    print("[2/2] 전체 의존성 + pretrained 설치")
    print(f"[log] 전체 로그: {setup_log}")
    print("-" * 72)
    if tail:
        print("\n".join(tail))
    else:
        print("설치 프로세스 시작 중...")


with setup_log.open("w", encoding="utf-8") as log_writer:
    process = subprocess.Popen(
        [sys.executable, "-u", str(setup_script)],
        stdout=log_writer,
        stderr=subprocess.STDOUT,
        text=True,
    )

    with setup_log.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as log_reader:
        while process.poll() is None:
            chunk = log_reader.read()
            if chunk:
                pending += chunk.replace("\r", "\n")
                parts = pending.split("\n")
                pending = parts.pop()
                for item in parts:
                    item = item.strip()
                    if item:
                        tail.append(item)
                render_tail()
            time.sleep(0.25)

        chunk = log_reader.read()
        if chunk:
            pending += chunk.replace("\r", "\n")

if pending:
    for item in pending.split("\n"):
        item = item.strip()
        if item:
            tail.append(item)

return_code = process.wait()
render_tail(final=True)

if return_code != 0:
    error_tail = setup_log.read_text(
        encoding="utf-8",
        errors="replace",
    ).replace("\r", "\n").splitlines()[-80:]
    clear_output(wait=True)
    print("\n".join(error_tail))
    raise RuntimeError(
        f"setup script failed with exit code {return_code}. "
        f"전체 로그: {setup_log}"
    )

print("\n[OK] 설치 완료", flush=True)


# Kaggle working disk 절약: 설치에만 필요한 다운로드/패키지 캐시 제거
print("[cleanup] Kaggle 설치 캐시 정리", flush=True)
cache_dirs = [
    Path("/kaggle/tmp/gpt_sovits_downloads"),
    Path("/kaggle/tmp/huggingface/hub"),
    Path("/root/.cache/pip"),
    Path("/root/.cache/uv"),
]
for cache_dir in cache_dirs:
    if cache_dir.exists():
        try:
            shutil.rmtree(cache_dir)
            print("  removed:", cache_dir)
        except OSError as exc:
            print("  warn:", cache_dir, exc)

for disk_path in ("/kaggle/tmp", "/kaggle/working"):
    usage = shutil.disk_usage(disk_path)
    print(
        f"[disk] {disk_path}: "
        f"{usage.free / 1024**3:.2f} GiB free / "
        f"{usage.total / 1024**3:.2f} GiB total"
    )


## 2. Preflight 및 대조 실험

Git launcher가 Dataset 탐색과 Preflight를 실행한 뒤 headless 실험을 시작합니다. WebUI 공개 링크를 만들지 않습니다.


전처리 결과를 두 모델이 공유하고 서로 다른 새 학습 디렉터리를 사용합니다.


## 3. 첫 실험: GPT/S1 Grokfast OFF vs ON

두 T4에 각각 독립적인 FP32 S1 학습을 배치합니다. BERT ON, DPO OFF, S2 V5dev 고정. 세부 설정과 시간 예산은 Git 코드에서 관리합니다. 결과는 `/kaggle/working/grokking_*.zip`으로 저장합니다. 실패하거나 시간 제한에 도달해도 로그를 보존합니다.


## 실행

이 셀은 Git launcher에 연구 모드만 전달합니다. 이후 실험 변경은 Git에서 관리합니다.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_DIR = Path("/kaggle/tmp/GPT-SoVITS")
LAUNCHER = REPO_DIR / "tools" / "kaggle_launch_webui.py"
env = os.environ.copy()
env["GPT_SOVITS_KAGGLE_GROKKING"] = "1"

subprocess.run(
    [sys.executable, "-u", str(LAUNCHER)],
    cwd=REPO_DIR,
    env=env,
    check=True,
)
